# Manufacturing Defect Detection — EDA Notebook
**Week 2 Checkpoint | INTELSYS Final Project AY2526**

**Team Members:** Doton, John Harold D. | Baguio, Eriel Ben L. | Bernarte, Karl Shane Y. | De Castro, Juan Carlo C.

This notebook covers:
1. Dataset loading and inspection
2. Class distribution analysis
3. Image statistics (dimensions, pixel value distributions)
4. Sample visualization
5. Train/Val/Test split finalization
6. Summary statistics for Model Card

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Reproducibility seed — always set this before any random operations
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Libraries loaded successfully.')

## 1. Dataset Loading

We use the **Bridge Cracks Image Dataset** from Kaggle (yidazhang07/bridge-cracks-image).
Expected structure after download:
```
data/
  crack/       ← images with surface crack defects
  no_crack/    ← normal (defect-free) images
```

In [ ]:
# ── Dataset Path ──────────────────────────────────────────────────────────────
# Adjust DATA_ROOT to where you placed the Kaggle dataset
DATA_ROOT = Path('data')  # Change this if your path differs

# Discover class folders automatically
# Each subfolder is treated as one class label
class_dirs = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir()])
print(f'Classes found: {[d.name for d in class_dirs]}')

# Build a flat DataFrame: filepath | label
records = []
for class_dir in class_dirs:
    for img_path in class_dir.glob('*.[jp][pn]g'):  # jpg + png
        records.append({'filepath': str(img_path), 'label': class_dir.name})

df = pd.DataFrame(records)
print(f'Total images found: {len(df)}')
df.head()

## 2. Class Distribution

In [ ]:
# ── Class Distribution ────────────────────────────────────────────────────────
# Understanding class balance is critical before training.
# Severe imbalance may require weighted loss or oversampling.

class_counts = df['label'].value_counts()
print('Class counts:')
print(class_counts)

imbalance_ratio = class_counts.max() / class_counts.min()
print(f'\nImbalance ratio (max/min): {imbalance_ratio:.2f}')
if imbalance_ratio > 3:
    print('⚠ Significant imbalance detected — consider weighted CrossEntropyLoss or SMOTE.')
else:
    print('✓ Class balance acceptable.')

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#E74C3C' if 'crack' in c.lower() else '#2ECC71' for c in class_counts.index]
class_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Image Count')
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('outputs/class_distribution.png', dpi=150)
plt.show()

## 3. Image Dimension Analysis

In [ ]:
# ── Image Dimensions ──────────────────────────────────────────────────────────
# CNNs expect consistent input sizes. We check if images need resizing.
# Sampling 500 images for speed; adjust sample_n for full scan.

os.makedirs('outputs', exist_ok=True)

sample_n = min(500, len(df))
sample_df = df.sample(n=sample_n, random_state=SEED)

widths, heights, channels = [], [], []
for _, row in sample_df.iterrows():
    try:
        img = Image.open(row['filepath'])
        w, h = img.size
        c = len(img.getbands())  # 3 = RGB, 1 = grayscale
        widths.append(w)
        heights.append(h)
        channels.append(c)
    except Exception:
        pass  # Skip corrupt files

print(f'Width  — min:{min(widths)}, max:{max(widths)}, mean:{np.mean(widths):.0f}')
print(f'Height — min:{min(heights)}, max:{max(heights)}, mean:{np.mean(heights):.0f}')
print(f'Channels — distribution: {Counter(channels)}')

# Scatter of w vs h
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(widths, heights, alpha=0.3, s=10, c='steelblue')
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Height (px)')
axes[0].set_title('Image Dimensions Scatter')

axes[1].hist(widths, bins=30, color='steelblue', alpha=0.7, label='Width')
axes[1].hist(heights, bins=30, color='salmon', alpha=0.7, label='Height')
axes[1].set_xlabel('Pixels')
axes[1].set_ylabel('Count')
axes[1].set_title('Width / Height Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/image_dimensions.png', dpi=150)
plt.show()

## 4. Pixel Intensity Distribution

In [ ]:
# ── Pixel Statistics ──────────────────────────────────────────────────────────
# Computing mean and std per channel is required to build the
# normalization transform for PyTorch (transforms.Normalize(mean, std)).

TARGET_SIZE = (224, 224)  # ResNet/MobileNet standard input
SAMPLE_STAT = min(300, len(df))

pixel_means = {cls: [] for cls in df['label'].unique()}
all_pixels = []

for _, row in df.sample(SAMPLE_STAT, random_state=SEED).iterrows():
    try:
        img = Image.open(row['filepath']).convert('RGB').resize(TARGET_SIZE)
        arr = np.array(img) / 255.0  # Normalize to [0,1]
        pixel_means[row['label']].append(arr.mean(axis=(0,1)))  # Per-channel mean
        all_pixels.append(arr.reshape(-1, 3))
    except Exception:
        pass

all_pixels = np.vstack(all_pixels)  # shape: (N*224*224, 3)
DATASET_MEAN = all_pixels.mean(axis=0)
DATASET_STD  = all_pixels.std(axis=0)

print(f'Dataset mean (R,G,B): {DATASET_MEAN.round(4)}')
print(f'Dataset std  (R,G,B): {DATASET_STD.round(4)}')
print('Use these values in transforms.Normalize().')

# Histogram of R,G,B channels
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
channel_names = ['Red', 'Green', 'Blue']
channel_colors = ['red', 'green', 'blue']
for i, (name, color) in enumerate(zip(channel_names, channel_colors)):
    axes[i].hist(all_pixels[:, i], bins=50, color=color, alpha=0.6)
    axes[i].axvline(DATASET_MEAN[i], color='black', linestyle='--', label=f'mean={DATASET_MEAN[i]:.3f}')
    axes[i].set_title(f'{name} Channel')
    axes[i].set_xlabel('Pixel Value (0-1)')
    axes[i].legend()
plt.suptitle('Pixel Intensity Distribution per Channel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/pixel_distributions.png', dpi=150)
plt.show()

## 5. Sample Image Grid

In [ ]:
# ── Sample Visualization ──────────────────────────────────────────────────────
# Visually inspect examples of each class to spot label noise.

COLS = 5
classes = df['label'].unique()
fig, axes = plt.subplots(len(classes), COLS, figsize=(COLS*3, len(classes)*3))
if len(classes) == 1:
    axes = [axes]  # Ensure iterable

for row_i, cls in enumerate(classes):
    cls_samples = df[df['label'] == cls].sample(n=COLS, random_state=SEED)
    for col_i, (_, sample_row) in enumerate(cls_samples.iterrows()):
        ax = axes[row_i][col_i]
        try:
            img = Image.open(sample_row['filepath']).convert('RGB')
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'Load Error', ha='center')
        ax.axis('off')
        if col_i == 0:
            ax.set_ylabel(cls, fontsize=12, fontweight='bold', rotation=0,
                          labelpad=60, va='center')

plt.suptitle('Sample Images per Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/sample_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Train / Validation / Test Split

In [ ]:
# ── Train/Val/Test Split ──────────────────────────────────────────────────────
# Standard split: 70% train | 15% val | 15% test
# Stratified to preserve class distribution in all splits.

from sklearn.model_selection import train_test_split

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15  # Must sum to 1

# Step 1: Split off test set (15%)
df_train_val, df_test = train_test_split(
    df, test_size=TEST_RATIO, stratify=df['label'], random_state=SEED
)

# Step 2: Split remaining into train + val
val_size_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
df_train, df_val = train_test_split(
    df_train_val, test_size=val_size_adjusted, stratify=df_train_val['label'], random_state=SEED
)

# Tag each row with its split
df_train = df_train.copy(); df_train['split'] = 'train'
df_val   = df_val.copy();   df_val['split']   = 'val'
df_test  = df_test.copy();  df_test['split']  = 'test'
df_splits = pd.concat([df_train, df_val, df_test])

# Save for downstream scripts
df_splits.to_csv('outputs/dataset_splits.csv', index=False)

print('Split summary:')
split_summary = df_splits.groupby(['split','label']).size().unstack(fill_value=0)
print(split_summary)
print(f'\nTotal — Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}')

# Visual split overview
split_totals = df_splits['split'].value_counts()[['train','val','test']]
fig, ax = plt.subplots(figsize=(6, 4))
split_totals.plot(kind='bar', ax=ax, color=['#3498DB','#F39C12','#2ECC71'], edgecolor='black')
ax.set_title('Dataset Split Sizes')
ax.set_ylabel('Image Count')
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('outputs/split_distribution.png', dpi=150)
plt.show()

## 7. EDA Summary

In [ ]:
# ── EDA Summary Table ─────────────────────────────────────────────────────────
print('=' * 55)
print('  EDA SUMMARY — Manufacturing Defect Detection')
print('=' * 55)
print(f'  Total images       : {len(df)}')
print(f'  Classes            : {list(df["label"].unique())}')
print(f'  Imbalance ratio    : {imbalance_ratio:.2f}')
print(f'  Typical image size : ~{int(np.mean(widths))}x{int(np.mean(heights))} px')
print(f'  Resize target      : {TARGET_SIZE}')
print(f'  Dataset mean (RGB) : {DATASET_MEAN.round(4)}')
print(f'  Dataset std  (RGB) : {DATASET_STD.round(4)}')
print(f'  Train / Val / Test : {len(df_train)} / {len(df_val)} / {len(df_test)}')
print('=' * 55)
print('Outputs saved to outputs/')